# Phase 4 — Ablation Study
Tests 6 E(S) component variants to show each component contributes.

In [1]:
import pandas as pd
import numpy as np
import shap
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from scipy.stats import spearmanr
from deap import base, creator, tools
print('All imports successful')

e:\project1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [4]:
# Cell 2 — Configuration
BASE    = Path('E:/project1/EW_MOGA_Project')
DATA    = BASE / 'data' / 'Brain Tumor.csv'
RESULTS = BASE / 'results'
ABL_DIR = BASE / 'results' / 'ablation'

# Create directories if they don't exist
RESULTS.mkdir(exist_ok=True, parents=True)
ABL_DIR.mkdir(exist_ok=True, parents=True)

FEATURE_COLS = [
    'Mean', 'Variance', 'Standard Deviation', 'Entropy',
    'Skewness', 'Kurtosis', 'Contrast', 'Energy',
    'ASM', 'Homogeneity', 'Dissimilarity', 'Correlation', 'Coarseness'
]

N_FEATURES  = 13
POP_SIZE    = 100
N_GEN       = 20
CX_PROB     = 0.8
MUT_PROB    = 1.0 / N_FEATURES
N_FOLDS     = 5
SEEDS       = [0, 1, 2]
N_PERTURB   = 5
B_BOOTSTRAP = 2
N_ABLATION_SAMPLES = 10
TOP_K       = 5

ABLATION_VARIANTS = {
    'SCI_only'        : (True,  False, False),
    'FID_only'        : (False, True,  False),
    'LFS_only'        : (False, False, True),
    'SCI_FID'         : (True,  True,  False),
    'SCI_LFS'         : (True,  False, True),
    'FID_LFS'         : (False, True,  True),
    'Full_SCI_FID_LFS': (True,  True,  True),
}

CHECKPOINT_FILE_ABL = ABL_DIR / 'ablation_checkpoint.csv'

print('Configuration set')
print(f'Features  : {FEATURE_COLS}')
print(f'Data      : {DATA}')
print(f'Ablation  : {ABL_DIR}')
print(f'Checkpoint: {CHECKPOINT_FILE_ABL}')
print(f'Variants  : {list(ABLATION_VARIANTS.keys())}')

Configuration set
Features  : ['Mean', 'Variance', 'Standard Deviation', 'Entropy', 'Skewness', 'Kurtosis', 'Contrast', 'Energy', 'ASM', 'Homogeneity', 'Dissimilarity', 'Correlation', 'Coarseness']
Data      : E:\project1\EW_MOGA_Project\data\Brain Tumor.csv
Ablation  : E:\project1\EW_MOGA_Project\results\ablation
Checkpoint: E:\project1\EW_MOGA_Project\results\ablation\ablation_checkpoint.csv
Variants  : ['SCI_only', 'FID_only', 'LFS_only', 'SCI_FID', 'SCI_LFS', 'FID_LFS', 'Full_SCI_FID_LFS']


In [5]:
df    = pd.read_csv(DATA)
X_raw = df[FEATURE_COLS].values.astype(float)
y_raw = df['Class'].values.astype(int)
print(f'Dataset: {X_raw.shape}')

def preprocess(X_tr, X_val, y_tr, random_state=42):
    scaler   = MinMaxScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    sm = SMOTE(random_state=random_state)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_sc, y_tr)
    return X_tr_sm, X_val_sc, y_tr_sm

print('Dataset loaded and preprocessing defined')

Dataset: (3762, 13)
Dataset loaded and preprocessing defined


In [6]:
# ── All component functions (copied from Notebook 2) ─────────
_prev_ranking = [None]

def compute_sci(X_tr, y_tr, mask, seed=42):
    idx = [i for i,v in enumerate(mask) if v==1]
    if not idx: return 0.0
    rng = np.random.RandomState(seed)
    rankings = []
    for _ in range(B_BOOTSTRAP):
        boot_idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        X_b, y_b = X_tr[boot_idx], y_tr[boot_idx]
        rf = RandomForestClassifier(n_estimators=30, random_state=seed, n_jobs=-1)
        rf.fit(X_b[:, idx], y_b)
        bg = X_b[rng.choice(len(X_b), min(50,len(X_b)), replace=False)]
        exp = shap.TreeExplainer(rf, bg)
        sv  = exp.shap_values(X_b[:min(30,len(X_b)),:], check_additivity=False)
        if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
        else: sv_arr = np.abs(sv)
        if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
        rankings.append(sv_arr.mean(axis=0))
    rhos = []
    for i, j in combinations(range(len(rankings)), 2):
        rho, _ = spearmanr(rankings[i], rankings[j])
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos)) if rhos else 0.0

def compute_fid(X_tr, y_tr, mask, seed=42):
    idx = [i for i,v in enumerate(mask) if v==1]
    if not idx: return 1.0
    rf = RandomForestClassifier(n_estimators=30, random_state=seed, n_jobs=-1)
    rf.fit(X_tr[:, idx], y_tr)
    rng = np.random.RandomState(seed)
    bg  = X_tr[rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
    exp = shap.TreeExplainer(rf, bg)
    sv  = exp.shap_values(X_tr[:min(30,len(X_tr)),:], check_additivity=False)
    if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else: sv_arr = np.abs(sv)
    if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
    curr_rank = np.argsort(-sv_arr.mean(axis=0))
    k = min(TOP_K, len(idx))
    curr_top = set(curr_rank[:k])
    if _prev_ranking[0] is None:
        _prev_ranking[0] = curr_top
        return 0.0
    prev_top = _prev_ranking[0]
    _prev_ranking[0] = curr_top
    inter = len(curr_top & prev_top)
    union = len(curr_top | prev_top)
    return 1.0 - (inter/union) if union > 0 else 0.0

def compute_lfs(rf, X_val, idx, seed=42):
    if len(X_val) == 0: return 0.0
    rng = np.random.RandomState(seed)
    sample_idx = rng.choice(len(X_val), min(N_ABLATION_SAMPLES,len(X_val)), replace=False)
    X_s = X_val[sample_idx][:, idx]
    bg  = X_s[:min(20,len(X_s))]
    exp = shap.TreeExplainer(rf, bg)
    sv  = exp.shap_values(X_s, check_additivity=False)
    if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else: sv_arr = np.abs(sv)
    if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
    importance_order = np.argsort(-sv_arr.mean(axis=0))
    base_proba = rf.predict_proba(X_s)
    drops = []
    X_ablated = X_s.copy()
    for feat_idx in importance_order:
        X_ablated[:, feat_idx] = 0.0
        new_proba = rf.predict_proba(X_ablated)
        drops.append(np.mean(np.abs(base_proba - new_proba)))
    return float(np.clip(np.mean(drops)*10, 0, 1))

def compute_es(X_tr, y_tr, X_val, mask, rf, idx,
               use_sci=True, use_fid=True, use_lfs=True, seed=42):
    components = []
    if use_sci: components.append(compute_sci(X_tr, y_tr, mask, seed))
    if use_fid: components.append(1.0 - compute_fid(X_tr, y_tr, mask, seed))
    if use_lfs: components.append(compute_lfs(rf, X_val, idx, seed))
    return float(np.mean(components)) if components else 0.0

def compute_ccc(X_tr, y_tr, X_val, idx, seed=42):
    import xgboost as xgb
    from sklearn.svm import SVC
    from sklearn.neural_network import MLPClassifier
    classifiers = {
        'RF' : RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
        'XGB': xgb.XGBClassifier(n_estimators=100, random_state=seed,
                                  eval_metric='logloss', verbosity=0),
        'SVM': SVC(kernel='rbf', probability=True, random_state=seed),
        'MLP': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed),
    }
    rankings = []
    rng = np.random.RandomState(seed)
    for name, clf in classifiers.items():
        try:
            clf.fit(X_tr[:, idx], y_tr)
            if name in ['RF','XGB']: exp = shap.TreeExplainer(clf)
            else:
                bg = X_tr[:, idx][rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
                exp = shap.KernelExplainer(clf.predict_proba, bg)
            sv = exp.shap_values(X_val[:min(30,len(X_val)), :][:, idx], check_additivity=False)
            if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
            else: sv_arr = np.abs(sv)
            if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
            rankings.append(sv_arr.mean(axis=0))
        except: pass
    if len(rankings) < 2: return 0.0
    rhos = [spearmanr(rankings[i], rankings[j])[0]
            for i,j in combinations(range(len(rankings)),2)]
    return float(np.mean([r if not np.isnan(r) else 0.0 for r in rhos]))

def compute_pr(X_tr, y_tr, idx, seed=42):
    rf_b = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_b.fit(X_tr[:, idx], y_tr)
    rng = np.random.RandomState(seed)
    bg  = X_tr[rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
    exp = shap.TreeExplainer(rf_b, bg)
    sv  = exp.shap_values(X_tr[:min(30,len(X_tr)),:], check_additivity=False)
    if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else: sv_arr = np.abs(sv)
    if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
    base_rank = sv_arr.mean(axis=0)
    rhos = []
    for p in range(N_PERTURB):
        boot_idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        X_b, y_b = X_tr[boot_idx], y_tr[boot_idx]
        rf_p = RandomForestClassifier(n_estimators=100, random_state=seed+p, n_jobs=-1)
        rf_p.fit(X_b[:, idx], y_b)
        bg_p = X_b[rng.choice(len(X_b), min(50,len(X_b)), replace=False)]
        exp_p = shap.TreeExplainer(rf_p, bg_p)
        sv_p  = exp_p.shap_values(X_b[:min(30,len(X_b)),:], check_additivity=False)
        if isinstance(sv_p, list): sv_p_arr = np.mean([np.abs(r) for r in sv_p], axis=0)
        else: sv_p_arr = np.abs(sv_p)
        if sv_p_arr.ndim == 3: sv_p_arr = sv_p_arr.mean(axis=2)
        rho, _ = spearmanr(base_rank, sv_p_arr.mean(axis=0))
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos))

print('All component functions defined')

All component functions defined


In [7]:
# ── NSGA-II with configurable E(S) ───────────────────────────
for attr in ['FitnessAbl','IndAbl']:
    if attr in dir(creator): delattr(creator, attr)
creator.create('FitnessAbl', base.Fitness, weights=(1.0, 1.0))
creator.create('IndAbl', list, fitness=creator.FitnessAbl)

def make_ind_abl():
    ind = [np.random.randint(0,2) for _ in range(N_FEATURES)]
    if sum(ind) == 0: ind[np.random.randint(N_FEATURES)] = 1
    return creator.IndAbl(ind)

def mut_flip(individual, indpb):
    for i in range(len(individual)):
        if np.random.random() < indpb: individual[i] = 1 - individual[i]
    if sum(individual) == 0: individual[np.random.randint(N_FEATURES)] = 1
    return individual,

tb_abl = base.Toolbox()
tb_abl.register('individual', make_ind_abl)
tb_abl.register('population', tools.initRepeat, list, tb_abl.individual)
tb_abl.register('mate',   tools.cxTwoPoint)
tb_abl.register('mutate', mut_flip, indpb=MUT_PROB)
tb_abl.register('select', tools.selNSGA2)

def run_ablation_variant(X_tr, y_tr, X_val, y_val,
                          use_sci, use_fid, use_lfs, seed=42):
    np.random.seed(seed)
    _prev_ranking[0] = None

    def eval_ind(ind):
        idx = [i for i,v in enumerate(ind) if v==1]
        if not idx: idx = [0]
        rf = RandomForestClassifier(n_estimators=30, random_state=seed, n_jobs=-1)
        rf.fit(X_tr[:, idx], y_tr)
        acc = accuracy_score(y_val, rf.predict(X_val[:, idx]))
        es  = compute_es(X_tr, y_tr, X_val, list(ind), rf, idx,
                         use_sci, use_fid, use_lfs, seed)
        return acc, es

    tb_abl.register('evaluate', eval_ind)
    pop  = tb_abl.population(n=POP_SIZE)
    fits = list(map(tb_abl.evaluate, pop))
    for ind, fit in zip(pop, fits): ind.fitness.values = fit
    pop = tb_abl.select(pop, len(pop))

    for gen in range(N_GEN):
        offspring = tools.selTournamentDCD(pop, len(pop))
        offspring = [tb_abl.clone(o) for o in offspring]
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if np.random.random() < CX_PROB:
                tb_abl.mate(c1, c2)
                del c1.fitness.values, c2.fitness.values
        for mut in offspring:
            tb_abl.mutate(mut)
            if not mut.fitness.valid: del mut.fitness.values
        invalid = [ind for ind in offspring if not ind.fitness.valid]
        fits    = list(map(tb_abl.evaluate, invalid))
        for ind, fit in zip(invalid, fits): ind.fitness.values = fit
        pop = tb_abl.select(pop + offspring, POP_SIZE)

    pareto = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
    best   = max(pareto, key=lambda ind: sum(ind.fitness.values))
    idx    = [i for i,v in enumerate(best) if v==1]
    if not idx: idx = [0]
    rf_f = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_f.fit(X_tr[:, idx], y_tr)
    acc = accuracy_score(y_val, rf_f.predict(X_val[:, idx]))
    ccc = compute_ccc(X_tr, y_tr, X_val, idx, seed)
    pr  = compute_pr(X_tr, y_tr, idx, seed)
    return acc, len(idx), ', '.join([FEATURE_COLS[i] for i in idx]), ccc, pr

print('Ablation variant runner defined')

Ablation variant runner defined


In [9]:
# ── Run all ablation variants with checkpoint saving ──────────
CHECKPOINT_FILE_ABL = ABL_DIR / 'ablation_checkpoint.csv'

# Resume from checkpoint if exists
if CHECKPOINT_FILE_ABL.exists():
    df_existing = pd.read_csv(CHECKPOINT_FILE_ABL)
    all_results = df_existing.to_dict('records')
    completed   = set(zip(df_existing['variant'],
                          df_existing['seed'],
                          df_existing['fold']))
    print(f'Resuming from checkpoint — {len(all_results)} runs already done')
    print(f'Completed variants:')
    print(df_existing.variant.value_counts().to_dict())
else:
    all_results = []
    completed   = set()
    print('Starting fresh — no checkpoint found')

total_variants = len(ABLATION_VARIANTS)
total_runs     = total_variants * len(SEEDS) * N_FOLDS
run_num        = len(all_results)

for v_num, (variant_name, (use_sci, use_fid, use_lfs)) in enumerate(ABLATION_VARIANTS.items()):
    print(f'\n[{v_num+1}/{total_variants}] Variant: {variant_name} '
          f'(SCI={use_sci}, FID={use_fid}, LFS={use_lfs})')

    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_raw, y_raw)):

            # Skip already completed runs
            if (variant_name, seed, fold) in completed:
                print(f'  [SKIP] {variant_name} seed={seed} fold={fold} already done')
                continue

            run_num += 1
            print(f'  [{run_num}/{total_runs}] {variant_name} seed={seed} fold={fold}',
                  end=' ... ')
            t0 = time.time()

            X_tr,  X_val  = X_raw[tr_idx],  X_raw[val_idx]
            y_tr,  y_val  = y_raw[tr_idx],  y_raw[val_idx]
            X_tr_pp, X_val_pp, y_tr_pp = preprocess(X_tr, X_val, y_tr, seed)

            acc, n_feat, feat_names, ccc, pr = run_ablation_variant(
                X_tr_pp, y_tr_pp, X_val_pp, y_val,
                use_sci, use_fid, use_lfs, seed)
            elapsed = time.time() - t0

            all_results.append({
                'variant'      : variant_name,
                'use_sci'      : use_sci,
                'use_fid'      : use_fid,
                'use_lfs'      : use_lfs,
                'seed'         : seed,
                'fold'         : fold,
                'accuracy'     : acc,
                'n_features'   : n_feat,
                'runtime_sec'  : elapsed,
                'feature_names': feat_names,
                'audit_ccc'    : ccc,
                'audit_pr'     : pr,
            })

            # Save checkpoint after every single run
            pd.DataFrame(all_results).to_csv(CHECKPOINT_FILE_ABL, index=False)
            print(f'acc={acc:.4f} ccc={ccc:.3f} pr={pr:.3f} [{elapsed:.0f}s] ✓ saved')
            completed.add((variant_name, seed, fold))

df_abl = pd.DataFrame(all_results)
df_abl.to_csv(ABL_DIR / 'ablation_results.csv', index=False)
print(f'\nAll {len(df_abl)} runs complete')
print(f'Saved: {ABL_DIR}/ablation_results.csv')
print()
print('='*60)
print('ABLATION STUDY SUMMARY')
print('='*60)
summary = df_abl.groupby('variant').agg(
    accuracy_mean=('accuracy',  'mean'),
    accuracy_std =('accuracy',  'std'),
    ccc_mean     =('audit_ccc', 'mean'),
    ccc_std      =('audit_ccc', 'std'),
    pr_mean      =('audit_pr',  'mean'),
    pr_std       =('audit_pr',  'std'),
    n_feat_mean  =('n_features','mean'),
).round(4)
print(summary.to_string())
print()
print('Notebook 4 complete — ready to run Notebook 5')

Starting fresh — no checkpoint found

[1/7] Variant: SCI_only (SCI=True, FID=False, LFS=False)
  [1/105] SCI_only seed=0 fold=0 ... 

C:\Users\MAHNOOR\AppData\Local\Temp\ipykernel_23268\518592449.py:23: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(rankings[i], rankings[j])
C:\Users\MAHNOOR\AppData\Local\Temp\ipykernel_23268\518592449.py:23: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(rankings[i], rankings[j])
C:\Users\MAHNOOR\AppData\Local\Temp\ipykernel_23268\518592449.py:23: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(rankings[i], rankings[j])
C:\Users\MAHNOOR\AppData\Local\Temp\ipykernel_23268\518592449.py:23: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(rankings[i], rankings[j])
e:\project1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.

acc=0.9907 ccc=0.333 pr=0.975 [299s] ✓ saved
  [2/105] SCI_only seed=0 fold=1 ... 

e:\project1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
100%|██████████| 30/30 [00:00<00:00, 455.23it/s]


acc=0.9907 ccc=0.550 pr=0.947 [395s] ✓ saved
  [3/105] SCI_only seed=0 fold=2 ... 

C:\Users\MAHNOOR\AppData\Local\Temp\ipykernel_23268\518592449.py:23: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(rankings[i], rankings[j])
e:\project1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
100%|██████████| 30/30 [00:00<00:00, 144.40it/s]


... (201 outputs left out)

e:\project1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
100%|██████████| 30/30 [00:00<00:00, 205.60it/s]


acc=0.9920 ccc=0.649 pr=0.963 [721s] ✓ saved
  [105/105] Full_SCI_FID_LFS seed=2 fold=4 ... 

e:\project1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
100%|██████████| 30/30 [00:00<00:00, 468.14it/s]


acc=0.9880 ccc=1.000 pr=0.998 [555s] ✓ saved

All 105 runs complete
Saved: E:\project1\EW_MOGA_Project\results\ablation/ablation_results.csv

ABLATION STUDY SUMMARY
                  accuracy_mean  accuracy_std  ccc_mean  ccc_std  pr_mean  pr_std  n_feat_mean
variant                                                                                       
FID_LFS                  0.9903        0.0034    0.6256   0.0985   0.9375  0.0636       7.0000
FID_only                 0.9901        0.0035    0.6057   0.1126   0.9639  0.0392       7.2000
Full_SCI_FID_LFS         0.9903        0.0028    0.7220   0.1301   0.9679  0.0314       6.3333
LFS_only                 0.9901        0.0030    0.6088   0.1189   0.9532  0.0426       7.0667
SCI_FID                  0.9904        0.0030    0.7013   0.1887   0.9670  0.0315       6.0000
SCI_LFS                  0.9904        0.0031    0.6119   0.1882   0.9711  0.0316       5.0000
SCI_only                 0.9902        0.0027    0.6382   0.2041   0.9733  

In [10]:
# ── Save and summarize ablation results ──────────────────────
out_path = RESULTS / 'ablation_results.csv'
df_abl.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print()
print('='*65)
print('ABLATION STUDY SUMMARY')
print('='*65)
summary = df_abl.groupby('variant').agg(
    accuracy_mean=('accuracy','mean'),
    accuracy_std =('accuracy','std'),
    ccc_mean     =('audit_ccc','mean'),
    ccc_std      =('audit_ccc','std'),
    pr_mean      =('audit_pr','mean'),
    pr_std       =('audit_pr','std'),
    n_feat_mean  =('n_features','mean'),
).round(4)
print(summary.to_string())
print()
print('Notebook 4 complete — ready to run Notebook 5')
print('='*65)

Saved: E:\project1\EW_MOGA_Project\results\ablation_results.csv

ABLATION STUDY SUMMARY
                  accuracy_mean  accuracy_std  ccc_mean  ccc_std  pr_mean  pr_std  n_feat_mean
variant                                                                                       
FID_LFS                  0.9903        0.0034    0.6256   0.0985   0.9375  0.0636       7.0000
FID_only                 0.9901        0.0035    0.6057   0.1126   0.9639  0.0392       7.2000
Full_SCI_FID_LFS         0.9903        0.0028    0.7220   0.1301   0.9679  0.0314       6.3333
LFS_only                 0.9901        0.0030    0.6088   0.1189   0.9532  0.0426       7.0667
SCI_FID                  0.9904        0.0030    0.7013   0.1887   0.9670  0.0315       6.0000
SCI_LFS                  0.9904        0.0031    0.6119   0.1882   0.9711  0.0316       5.0000
SCI_only                 0.9902        0.0027    0.6382   0.2041   0.9733  0.0301       5.4667

Notebook 4 complete — ready to run Notebook 5
